In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

path = '../data/ERA5/reanalysis-era5-single-levels-timeseries-sfc6369p5v8.nc'
ds = xr.open_dataset(path)

u10 = ds['u10'].values.astype(float)
v10 = ds['v10'].values.astype(float)
t2m = ds['t2m'].values.astype(float)
sst = ds['sst'].values.astype(float)
ssrd = ds['ssrd'].values.astype(float)
strd = ds['strd'].values.astype(float)

U = np.sqrt(u10**2 + v10**2)
tau_x = 1.225 * 1.3e-3 * U * u10
tau_y = 1.225 * 1.3e-3 * U * v10
QT = 15.0 * (t2m - sst) + 1e-6 * ssrd + 1e-6 * strd

X = np.column_stack([u10, v10, t2m, sst, ssrd, strd])
Y = np.column_stack([tau_x, tau_y, QT])
target_names = ['tau_x', 'tau_y', 'QT']

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
y_pred_lin = lin.predict(X_test)
rmse_lin = np.sqrt(((y_pred_lin - y_test) ** 2).mean(axis=0))

X_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train)
X_train_s = X_scaler.transform(X_train)
X_test_s = X_scaler.transform(X_test)
y_train_s = y_scaler.transform(y_train)

mlp = MLPRegressor(hidden_layer_sizes=(32, 32), activation='relu', random_state=42, max_iter=5000, early_stopping=True)
mlp.fit(X_train_s, y_train_s)
y_pred_mlp = y_scaler.inverse_transform(mlp.predict(X_test_s))
rmse_mlp = np.sqrt(((y_pred_mlp - y_test) ** 2).mean(axis=0))

results = pd.DataFrame({'target': target_names, 'linear_rmse': rmse_lin, 'mlp_rmse': rmse_mlp})
results

In [ ]:
plt.figure(figsize=(8, 4))
x = np.arange(len(target_names))
width = 0.35
plt.bar(x - width/2, rmse_lin, width, label='Linear')
plt.bar(x + width/2, rmse_mlp, width, label='MLP')
plt.xticks(x, target_names)
plt.ylabel('RMSE')
plt.title('Linear vs MLP RMSE')
plt.legend()
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(y_test[:, 0], label='Truth tau_x')
axes[0].plot(y_pred_lin[:, 0], label='Linear tau_x')
axes[0].plot(y_pred_mlp[:, 0], label='MLP tau_x')
axes[0].legend()
axes[0].set_title('tau_x predictions')

axes[1].plot(y_test[:, 2], label='Truth QT')
axes[1].plot(y_pred_lin[:, 2], label='Linear QT')
axes[1].plot(y_pred_mlp[:, 2], label='MLP QT')
axes[1].legend()
axes[1].set_title('QT predictions')
plt.tight_layout()